# Linked City Dashboard
Generates `dashboard_demo_newNew.html` from `dashboard_template_new.html`.

**Fixes for Plotly Python 6.x + Plotly.js 3.x:**
- `clickmode='event'` + `selected/unselected opacity=1` → prevents Plotly 3.x selection dimming
- `{{JS_CHART_DATA}}` token → embeds all color/id state as plain JS constants (avoids binary customdata encoding)
- `bindEvents()` with retry loop → waits for async `newPlot` before binding `.on()` listeners

In [2]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.colors import sample_colorscale

In [3]:
import folium

In [4]:
ROOT          = Path.cwd()
TEMPLATE_PATH = ROOT / "dashboard_template_new.html"
OUTPUT_PATH   = ROOT / "dashboard_demo_lab10.html"

In [6]:
def load_sample_data() -> pd.DataFrame:
    rows = [
        {"city_id": 0, "city": "Charlotte",  "state": "NC", "lat": 35.23, "lon": -80.84, "health_index": 81.3, "green_space": 10.6, "population": 879_000, "avg_temp": 61.0},
        {"city_id": 1, "city": "Raleigh",    "state": "NC", "lat": 35.78, "lon": -78.64, "health_index": 88.3, "green_space": 39.1, "population": 467_000, "avg_temp": 58.4},
        {"city_id": 2, "city": "Durham",     "state": "NC", "lat": 35.99, "lon": -78.90, "health_index": 80.6, "green_space": 35.0, "population": 274_000, "avg_temp": 60.1},
        {"city_id": 3, "city": "Greensboro", "state": "NC", "lat": 36.07, "lon": -79.79, "health_index": 76.0, "green_space": 16.4, "population": 290_000, "avg_temp": 59.2},
        {"city_id": 4, "city": "Atlanta",    "state": "GA", "lat": 33.75, "lon": -84.39, "health_index": 60.5, "green_space": 15.5, "population": 498_000, "avg_temp": 62.8},
        {"city_id": 5, "city": "Nashville",  "state": "TN", "lat": 36.16, "lon": -86.78, "health_index": 64.7, "green_space": 18.2, "population": 689_000, "avg_temp": 61.3},
        {"city_id": 6, "city": "Memphis",    "state": "TN", "lat": 35.15, "lon": -90.05, "health_index": 55.4, "green_space": 14.1, "population": 633_000, "avg_temp": 64.5},
        {"city_id": 7, "city": "Louisville", "state": "KY", "lat": 38.25, "lon": -85.76, "health_index": 79.8, "green_space": 18.7, "population": 633_000, "avg_temp": 57.1},
        {"city_id": 8, "city": "Richmond",   "state": "VA", "lat": 37.54, "lon": -77.44, "health_index": 74.2, "green_space": 27.5, "population": 231_000, "avg_temp": 59.0},
        {"city_id": 9, "city": "Washington", "state": "DC", "lat": 38.91, "lon": -77.04, "health_index": 82.4, "green_space": 22.4, "population": 712_000, "avg_temp": 58.0},
    ]
    df = pd.DataFrame(rows)
    df["city_label"] = df["city"] + ", " + df["state"]
    return df

In [7]:
def health_color(score: float) -> str:
    if score >= 80: return "#2BEEBD"
    if score >= 70: return "#F2F282"
    return "#ef4444"


# Shared trace properties: prevent Plotly 3.x from dimming non-selected points on click
NO_DIM = dict(
    selected=dict(marker=dict(opacity=1)),
    unselected=dict(marker=dict(opacity=1)),
)


def make_js_chart_data(df: pd.DataFrame) -> str:
    """
    Embed all chart state as a plain JS constant window.CHART_DATA.

    Why: Plotly Python 6.x serialises numeric arrays (including customdata)
    as base64 binary blobs in the HTML. Plotly.js 3.x cannot easily unpack
    these back to plain JS arrays from user code, so we side-step the issue
    entirely by writing the data we need directly into the page.
    """
    bar_df = df.sort_values("health_index", ascending=False)
    temp_min = df["avg_temp"].min()
    temp_max = df["avg_temp"].max()
    temp_range = temp_max - temp_min if temp_max != temp_min else 1
    scatter_colors = [
        sample_colorscale("Tealgrn", (v - temp_min) / temp_range)[0]
        for v in df["avg_temp"]
    ]
    scatter_sizes = ((df["population"] / 45000).round(1) + 8).tolist()

    data = {
        "mapBaseColors":     [health_color(s) for s in df["health_index"]],
        "barIds":            bar_df["city_id"].tolist(),
        "scatterBaseColors": scatter_colors,
        "scatterSizes":      scatter_sizes,
    }
    return f"<script>\n  window.CHART_DATA = {json.dumps(data)};\n</script>"

def make_map(df: pd.DataFrame) -> str:
    fig = go.Figure(
        go.Scattermapbox(
            lat=df["lat"],
            lon=df["lon"],
            text=df["city_label"],
            customdata=df[["city_id"]].to_numpy(),
            mode="markers+text",
            textposition="top right",
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Health=%{meta[0]:.1f}<br>"
                "Green Space=%{meta[1]:.1f}%<br>"
                "Population=%{meta[2]:,}<br>"
                "Avg Temp=%{meta[3]:.1f} F"
                "<extra></extra>"
            ),
            meta=df[["health_index", "green_space", "population", "avg_temp"]].to_numpy(),
            marker=dict(
                color=[health_color(s) for s in df["health_index"]],
                size=(df["population"] / 52000).round(1) + 8,
                opacity=0.95,
            ),
            **NO_DIM,
        )
    )
    fig.update_layout(
        title="City Locations and Health Scores",
        paper_bgcolor="#121a2b",
        plot_bgcolor="#121a2b",
        font=dict(color="#e5edf7", size=11),
        margin=dict(l=8, r=8, t=42, b=8),
        clickmode="event",  # prevent Plotly 3.x selection dimming
        mapbox=dict(
            style="carto-darkmatter",
            center=dict(lat=36.2, lon=-81.5),
            zoom=4.7,
        ),
    )
    return pio.to_html(fig, include_plotlyjs="cdn", full_html=False, div_id="map-chart")

In [8]:
mapbox_url = 'https://api.maptiler.com/maps/winter-v4/style.json?key=XL7pq33CzA3lwocBF57x'

In [10]:
def make_map(df: pd.DataFrame) -> str:
    fig = go.Figure(
        go.Scattermap(
            lat=df["lat"],
            lon=df["lon"],
            text=df["city_label"],
            customdata=df[["city_id"]].to_numpy(),
            mode="markers+text",
            textposition="top right",
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Health=%{meta[0]:.1f}<br>"
                "Green Space=%{meta[1]:.1f}%<br>"
                "Population=%{meta[2]:,}<br>"
                "Avg Temp=%{meta[3]:.1f} F"
                "<extra></extra>"
            ),
            meta=df[["health_index", "green_space", "population", "avg_temp"]].to_numpy(),
            marker=dict(
                color=[health_color(s) for s in df["health_index"]],
                size=(df["population"] / 52000).round(1) + 8,
                opacity=0.95,
            ),
            **NO_DIM,
        )
    )
    fig.update_layout(
        title="City Locations and Health Scores",
        paper_bgcolor="#121a2b",
        plot_bgcolor="#121a2b",
        font=dict(color="#e5edf7", size=10),
        margin=dict(l=8, r=8, t=42, b=8),
        clickmode="event",  # prevent Plotly 3.x selection dimming
        map=dict(
                style=mapbox_url,
                center=dict(lat=36.2, lon=-81.5),
                zoom=4.7
            )
                
    )
    return pio.to_html(fig, include_plotlyjs="cdn", full_html=False, div_id="map-chart")

In [11]:
def make_bar(df: pd.DataFrame) -> str:
    bar_df = df.sort_values("health_index", ascending=False)
    fig = go.Figure(
        go.Bar(
            x=bar_df["health_index"],
            y=bar_df["city_label"],
            orientation="h",
            customdata=bar_df[["city_id"]].to_numpy(),
            marker=dict(
                color=["#92F7B3"] * len(bar_df), #--> bar color
                line=dict(color="rgba(0,0,0,0)", width=0),
            ),
            text=bar_df["health_index"].map(lambda v: f"{v:.1f}"),
            textposition="inside", #--> Text value position
            hovertemplate="City=%{y}<br>Health=%{x:.1f}<extra></extra>",
            **NO_DIM,
        )
    )
    fig.update_layout(
        title="<b>Health Index by City</b>",
        paper_bgcolor="#121a2b",
        plot_bgcolor="#121a2b",
        font=dict(color="#e5edf7", size=12),
        margin=dict(l=10, r=32, t=32, b=18),
        xaxis=dict(title="Health Index", gridcolor="#334155", zeroline=False),
        yaxis=dict(title=None, gridcolor="#334155"),
        clickmode="event",  # prevent Plotly 3.x selection dimming
    )
    return pio.to_html(fig, include_plotlyjs=False, full_html=False, div_id="bar-chart")

In [12]:
def make_scatter(df: pd.DataFrame) -> str:
    temp_min = df["avg_temp"].min()
    temp_max = df["avg_temp"].max()
    temp_range = temp_max - temp_min if temp_max != temp_min else 1
    scatter_colors = [
        sample_colorscale("viridis", (v - temp_min) / temp_range)[0]
        for v in df["avg_temp"]
    ]
    fig = go.Figure(
        go.Scatter(
            x=df["green_space"],
            y=df["health_index"],
            text=df["city"],
            mode="markers+text",
            textposition="bottom center",
            customdata=df[["city_id"]].to_numpy(),
            hovertemplate=(
                "<b>%{meta[0]}</b><br>"
                "Green Space=%{x:.1f}%<br>"
                "Health=%{y:.1f}<br>"
                "Population=%{meta[1]:,}<br>"
                "Avg Temp=%{meta[2]:.1f} F"
                "<extra></extra>"
            ),
            meta=df[["city_label", "population", "avg_temp"]].to_numpy(),
            marker=dict(
                color=scatter_colors,
                size=(df["population"] / 45000).round(1) + 8,
                line=dict(color="white", width=2),
                opacity=0.95,
                sizemode="diameter",
            ),
            **NO_DIM,
        )
    )
    fig.update_layout(
        title="Green Space vs Health Index",
        paper_bgcolor="#121a2b",
        plot_bgcolor="#121a2b",
        font=dict(color="#e5edf7", size=9),
        margin=dict(l=18, r=18, t=42, b=18),
        xaxis=dict(title="Green Space (%)", gridcolor="#334155"),
        yaxis=dict(title="Health Index", gridcolor="#334155"),
        clickmode="event",  # prevent Plotly 3.x selection dimming
    )
    return pio.to_html(fig, include_plotlyjs=False, full_html=False, div_id="scatter-chart")

In [13]:
def make_table(df: pd.DataFrame) -> str:
    display_df = df[["city_id", "city_label", "population", "health_index", "green_space", "avg_temp"]].copy()
    display_df["population"]   = display_df["population"].map(lambda v: f"{v:,}")
    display_df["health_index"] = display_df["health_index"].map(lambda v: f"{v:.1f}")
    display_df["green_space"]  = display_df["green_space"].map(lambda v: f"{v:.1f}")
    display_df["avg_temp"]     = display_df["avg_temp"].map(lambda v: f"{v:.1f}")

    lines = [
        '<table class="summary-table">',
        "<thead><tr><th>City</th><th>Population</th><th>Health</th><th>Green Space (%)</th><th>Avg Temp (F)</th></tr></thead>",
        "<tbody>",
    ]
    for row in display_df.itertuples(index=False):
        lines.append(
            f'<tr data-city-id="{row.city_id}">'
            f"<td>{row.city_label}</td>"
            f"<td>{row.population}</td>"
            f"<td>{row.health_index}</td>"
            f"<td>{row.green_space}</td>"
            f"<td>{row.avg_temp}</td>"
            "</tr>"
        )
    lines.extend(["</tbody>", "</table>"])
    return "\n".join(lines)

In [14]:
def render_dashboard() -> str:
    df = load_sample_data()
    template = TEMPLATE_PATH.read_text(encoding="utf-8")
    replacements = {
        "{{PAGE_TITLE}}":         "Linked Dashboard Demo",
        "{{DASHBOARD_TITLE}}":    "Linked City Dashboard",
        "{{DASHBOARD_SUBTITLE}}": "Python builds the visuals, HTML/CSS handles layout, and JavaScript shares the selection state.",
        "{{HEADER_BADGE}}":       "Plotly + pandas + one shared city_id",
        "{{MAP_CHART}}":          make_map(df),
        "{{BAR_CHART}}":          make_bar(df),
        "{{SCATTER_CHART}}":      make_scatter(df),
        "{{TABLE_HTML}}":         make_table(df),
        "{{JS_CHART_DATA}}":      make_js_chart_data(df),
        "{{FOOTER_TEXT}}":        "Click any point, bar, or row to update the whole dashboard. Press Esc or double-click a chart to clear.",
    }
    html = template
    for token, value in replacements.items():
        html = html.replace(token, value)
    return html

In [15]:
def main() -> None:
    html = render_dashboard()
    OUTPUT_PATH.write_text(html, encoding="utf-8")
    print(f"Wrote {OUTPUT_PATH}")

main()

Wrote /Users/jayantabiswas/geovis_labs/lab10/Lab10_data/part2_linked_dashboard/dashboard_demo_lab10.html
